<a href="https://colab.research.google.com/github/ammar-aa/Fly_rank_internship_repo/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
"""
Section 1: Method choice and why

Method: Pairwise ranking via Logistic Regression on feature differences

There is no ground-truth label in this problem. The Week 4 baseline (score = -trend_pct * position_share) is itself a hand-written scoring rule built from two signals — trend_pct and gsc_sum_position — not a target to predict against. So this isn't classification or regression toward a known truth; it's a comparison between two different ways of scoring and ranking the same rows.

What actually matters for this problem is the order pages fall in, not the exact score value — the baseline's real job is to rank pages by refresh urgency. Pairwise ranking directly optimizes for "does page A outrank page B," which matches that goal more precisely than trying to hit an arbitrary numeric score.

To keep the comparison fair, the model uses the same two signals the baseline formula uses (trend_pct, gsc_sum_position) — no additional features, no leakage.

Logistic Regression is used because it's the simplest model that can learn a combination of the two signals. The baseline combines them multiplicatively with fixed, hand-picked weighting (-trend_pct * position_share). Training a Logistic Regression on pairwise feature differences tests whether a different, learned combination of the same two signals produces a meaningfully different — and possibly more sensible — ranking than the fixed formula.

Other menu methods don't fit as well here: clustering isn't appropriate since the goal isn't to discover groups, it's to compare two ranking systems; and Decision Tree / Random Forest / Gradient Boosting are heavier than needed for two features and would obscure the direct, interpretable weight comparison against the formula's fixed coefficients.
"""

'\nSection 1: Method choice and why\n\nMethod: Pairwise ranking via Logistic Regression on feature differences\n\nThere is no ground-truth label in this problem. The Week 4 baseline (score = -trend_pct * position_share) is itself a hand-written scoring rule built from two signals — trend_pct and gsc_sum_position — not a target to predict against. So this isn\'t classification or regression toward a known truth; it\'s a comparison between two different ways of scoring and ranking the same rows.\n\nWhat actually matters for this problem is the order pages fall in, not the exact score value — the baseline\'s real job is to rank pages by refresh urgency. Pairwise ranking directly optimizes for "does page A outrank page B," which matches that goal more precisely than trying to hit an arbitrary numeric score.\n\nTo keep the comparison fair, the model uses the same two signals the baseline formula uses (trend_pct, gsc_sum_position) — no additional features, no leakage.\n\nLogistic Regression 

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
"""
Section 2: Split design

Grouped by client, not time-aware.

Each content_hash_id appears exactly once in the dataset (verified: value_counts().max() == 1), so there is no repeated time series at the row level to be time-aware about. The Week 4 aggregation already collapsed the daily-grain source data into one summary row per page, with trend_pct and gsc_sum_position computed across the full time window per page. A time-aware split would be guarding against a leak that structurally cannot occur here, so it isn't used.

The real leak risk is client-level: multiple pages likely share the same client_hash_id, and if pages are split randomly, pages from the same client could land on both sides of train/test. Client-level effects (e.g. one client's whole site trending down for reasons unrelated to trend_pct or gsc_sum_position individually) could let the model partly learn "this client's pages behave a certain way" instead of the actual signal relationship being tested. Splitting by client_hash_id, so every page belonging to a given client stays entirely on one side, closes that leak.

Because this is a pairwise ranking setup, the split happens at the row level first, before pairs are generated: clients are divided into a train pool and a test pool, and only afterward are pairs sampled — separately — within each pool. This guarantees no single row, and no client, appears on both sides of any pair.
"""

'\nSection 2: Split design\n\nGrouped by client, not time-aware.\n\nEach content_hash_id appears exactly once in the dataset (verified: value_counts().max() == 1), so there is no repeated time series at the row level to be time-aware about. The Week 4 aggregation already collapsed the daily-grain source data into one summary row per page, with trend_pct and gsc_sum_position computed across the full time window per page. A time-aware split would be guarding against a leak that structurally cannot occur here, so it isn\'t used.\n\nThe real leak risk is client-level: multiple pages likely share the same client_hash_id, and if pages are split randomly, pages from the same client could land on both sides of train/test. Client-level effects (e.g. one client\'s whole site trending down for reasons unrelated to trend_pct or gsc_sum_position individually) could let the model partly learn "this client\'s pages behave a certain way" instead of the actual signal relationship being tested. Splittin

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
from google.colab import userdata
auth=userdata.get("HF_TOKEN")
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
con=duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{auth}'
);
""")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

In [4]:
df = con.sql(f"""
SELECT *
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [5]:
dfF = con.sql(f"""
SELECT SUM(gsc_impressions) AS gsc_impressions, client_hash_id, content_hash_id
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet'
GROUP BY client_hash_id, content_hash_id
""").df()

dfM = con.sql(f"""
SELECT SUM(gsc_impressions) AS gsc_impressions, client_hash_id, content_hash_id
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
GROUP BY client_hash_id, content_hash_id
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [6]:
df_trend = dfM.merge(dfF, on=['client_hash_id', 'content_hash_id'], suffixes=('_feb', '_mar'), how='outer')

In [7]:
df_trend = df_trend[df_trend['gsc_impressions_feb'] >= 30]
df_trend = df_trend[df_trend['gsc_impressions_mar'] > 0]

df_trend['trend_pct'] = (
    (df_trend['gsc_impressions_mar'] - df_trend['gsc_impressions_feb'])
    / df_trend['gsc_impressions_feb']
) * 100

clip_value = df_trend['trend_pct'].quantile(0.99)
df_trend['trend_pct'] = df_trend['trend_pct'].clip(lower=-clip_value, upper=clip_value)

In [8]:
df = df.groupby(['client_hash_id', 'content_hash_id'], as_index=False).agg(
    gsc_sum_position=('gsc_sum_position', 'sum'),
    gsc_avg_position=('gsc_avg_position', 'mean'),
)

In [9]:
df = df.merge(df_trend[['client_hash_id', 'content_hash_id', 'trend_pct']], on=['client_hash_id', 'content_hash_id'], how='left')

In [10]:
negative_mean = df.loc[df['trend_pct'] < 0, 'trend_pct'].mean()
positive_mean = df.loc[df['trend_pct'] > 0, 'trend_pct'].mean()
conditions = [
    df['trend_pct'] < negative_mean,
    (df['trend_pct'] < 0) & (df['trend_pct'] >= negative_mean),
    (df['trend_pct'] >= 0) & (df['trend_pct'] < positive_mean),   # now includes 0
    df['trend_pct'] >= positive_mean
]
ranks = ['Sharp decline', 'Mild decline', 'Mild growth', 'Strong growth']
df['trend_dir'] = np.select(conditions, ranks, default=None)

In [11]:
position_share = df['gsc_sum_position'] / df['gsc_sum_position'].sum()

cap_value = position_share.quantile(0.99)
position_share_capped = position_share.clip(upper=cap_value)

score = -df['trend_pct'] * position_share_capped * 1000
df['score']=score
df['score'] = df['score'] * 1000

In [12]:
df = df[['content_hash_id', 'client_hash_id', 'trend_pct', 'gsc_sum_position', 'score']].copy()

In [13]:
conditions = [
    (df['trend_pct'] < negative_mean) & (position_share > position_share.median()),
    (df['trend_pct'] < negative_mean) & (position_share <= position_share.median()),
    (df['trend_pct'] >= negative_mean) & (df['trend_pct'] < 0) & (position_share > position_share.median()),
    (df['trend_pct'] >= negative_mean) & (df['trend_pct'] < 0),
    (df['trend_pct'] >= 0) & (position_share > position_share.median()),
]
codes = [
    'strong declining trend with low page position',
    'strong declining trend',
    'mild declining trend with low page position',
    'mild declining trend',
    'low page position',
]
df['reason_code'] = np.select(conditions, codes, default='STABLE')


In [14]:
df['action'] = np.select(
    [
        df['reason_code'] == 'strong declining trend with low page position',
        df['reason_code'].isin(['strong declining trend', 'mild declining trend with low page position']),
    ],
    ['REFRESH', 'MONITOR'],
    default='SKIP'
)

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.